# Library vs custom codebook vector quantization on ResNet-18

This notebook compares two explicit codebook/vector-quantization implementations on the trained CIFAR-10 ResNet-18 checkpoint:

1. **Library baseline:** scikit-learn's established `KMeans` implementation constructs each tensor codebook.
2. **Custom implementation:** the project's mathematical NumPy k-means and nearest-codeword functions construct each tensor codebook.

TensorFlow Lite does not provide a built-in codebook/vector-quantization converter; its built-in PTQ is scalar INT8 quantization. Therefore, scikit-learn KMeans is used as the explicit inbuilt/library codebook baseline. Both paths use identical vector grouping, codebook size, tensor coverage, storage precision, and FP32 evaluation. No fake-quantization API is used.

In [1]:
from math import ceil, log2
from pathlib import Path
import gc
import json
import os
import sys
import time

import keras
import keras_hub
import numpy as np
import pandas as pd
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

import sklearn
import tensorflow as tf
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import CIFAR10_CLASS_NAMES
from src.quantization.custom_quantization import (
    CodebookQuantizedTensor,
    ModelCodebookQuantizationResult,
    codebook_vectorize_model,
    estimate_gradient_importance,
    quantization_mse,
    reconstruct_codebook_model_weights,
    vectorize_array,
)

np.random.seed(42)
tf.random.set_seed(42)
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TensorFlow: 2.20.0
Keras: 3.14.1
scikit-learn: 1.9.0


## 1. Configuration

This accuracy-oriented profile uses `VECTOR_DIM=1` and `CODEBOOK_SIZE=256`, so one 8-bit assignment represents one weight: a true 8-bit-per-weight codebook. The first convolution kernel and final classifier remain FP32. The custom path uses bounded gradient-sensitivity refinement; the scikit-learn path remains the unweighted 8-bit baseline.

In [2]:
MODEL_PATH = (
    PROJECT_ROOT / "artifacts" / "resnet18_cifar10_training"
    / "resnet18_cifar10_fp32.keras"
)
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "resnet18_codebook_vq_comparison"

VECTOR_DIM = 1
CODEBOOK_SIZE = 256
CODEBOOK_DTYPE = np.float32
KMEANS_ITERATIONS = 30
MAX_KMEANS_SAMPLES = 100_000
KMEANS_TOLERANCE = 1e-6
NUM_EVALUATION_SAMPLES = None
EVALUATION_BATCH_SIZE = 32
RANDOM_SEED = 42
PRESERVE_FIRST_AND_LAST_KERNELS = True
NUM_IMPORTANCE_CALIBRATION_SAMPLES = 256
IMPORTANCE_BATCH_SIZE = 16

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing trained model: {MODEL_PATH}. Run notebook 06 first.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSIGNMENT_BITS = max(1, ceil(log2(CODEBOOK_SIZE)))
print(f"Assignment bits per vector: {ASSIGNMENT_BITS}")
print(f"Assignment bits per scalar: {ASSIGNMENT_BITS / VECTOR_DIM:.2f}")

Assignment bits per vector: 8
Assignment bits per scalar: 8.00


## 2. Load the trained CNN and CIFAR-10 test set

In [3]:
model = keras.models.load_model(MODEL_PATH, compile=False)
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
train_labels = train_labels.reshape(-1).astype(np.int32)
test_labels = test_labels.reshape(-1).astype(np.int32)
if NUM_EVALUATION_SAMPLES is not None:
    test_images = test_images[:NUM_EVALUATION_SAMPLES]
    test_labels = test_labels[:NUM_EVALUATION_SAMPLES]

image_converter = keras_hub.layers.ResNetImageConverter(
    image_size=(224, 224),
    scale=[0.017124753831663668, 0.01750700280112045, 0.017429193899782133],
    offset=[-2.1179039301310043, -2.0357142857142856, -1.8044444444444445],
    interpolation="bicubic",
    crop_to_aspect_ratio=True,
)

def preprocess_images(images):
    return image_converter(images)

print(f"Loaded model: {MODEL_PATH}")
print(f"Evaluation images: {len(test_images):,}")
eligible_kernel_names = [
    getattr(weight, "path", weight.name)
    for weight in model.weights
    if np.issubdtype(weight.dtype, np.floating) and len(weight.shape) >= 2
]
PRESERVED_TENSOR_NAMES = (
    {eligible_kernel_names[0], eligible_kernel_names[-1]}
    if PRESERVE_FIRST_AND_LAST_KERNELS and eligible_kernel_names else set()
)

print("Preserved FP32 tensors:")
for name in sorted(PRESERVED_TENSOR_NAMES):
    print(f"  - {name}")
model.summary(expand_nested=True)

Loaded model: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/resnet18_cifar10_training/resnet18_cifar10_fp32.keras
Evaluation images: 10,000
Preserved FP32 tensors:
  - cifar10_predictions/kernel
  - conv1_conv/kernel


Model: "resnet18_cifar10_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ images (InputLayer)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ res_net_backbone                │ (None, 7, 7, 512)      │    11,186,112 │
│ (ResNetBackbone)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ input_layer (InputLayer)   │ (None, None, None, 3)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ conv1_pad (ZeroPadding2D)  │ (None, None, None, 3)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ conv1_conv (Conv2D)        │ (None, None, None, 64) │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ conv1_bn                   │ (None, None, None, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ conv1_relu (Activation)    │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ pool1_pad (ZeroPadding2D)  │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ pool1_pool (MaxPooling2D)  │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_1_conv       │ (None, None, None, 64) │        36,864 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_1_bn         │ (None, None, None, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_1_relu       │ (None, None, None, 64) │             0 │
│ (Activation)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_2_conv       │ (None, None, None, 64) │        36,864 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_2_bn         │ (None, None, None, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_add (Add)    │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block0_out          │ (None, None, None, 64) │             0 │
│ (Activation)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block1_1_conv       │ (None, None, None, 64) │        36,864 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block1_1_bn         │ (None, None, None, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│    └ stack0_block1_1_relu       │ (None, None, None, 64) │             0 │
│ (Activation)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 11,191,242 (42.69 MB)

 Trainable params: 11,181,642 (42.65 MB)

 Non-trainable params: 9,600 (37.50 KB)

## 3. Inbuilt/library codebook vectorization with scikit-learn

The wrapper below only adapts neural-network tensors into sub-vectors and records the result. Codebook fitting and assignment prediction are performed by scikit-learn `KMeans` and `pairwise_distances_argmin`.

In [4]:
def assignment_dtype(codebook_size):
    if codebook_size <= 256:
        return np.uint8
    if codebook_size <= 65536:
        return np.uint16
    return np.uint32

def sklearn_vector_quantize_array(
    array, *, name, vector_dim, codebook_size, codebook_dtype,
    max_samples, iterations, tolerance, seed
):
    vectors, layout = vectorize_array(array, vector_dim, axis=-1)
    flat_vectors = vectors.reshape(-1, vector_dim)
    rng = np.random.default_rng(seed)
    if max_samples is not None and len(flat_vectors) > max_samples:
        fit_vectors = flat_vectors[rng.choice(
            len(flat_vectors), size=max_samples, replace=False
        )]
    else:
        fit_vectors = flat_vectors

    estimator = KMeans(
        n_clusters=codebook_size,
        init="k-means++",
        n_init=1,
        max_iter=iterations,
        tol=tolerance,
        random_state=seed,
        algorithm="lloyd",
    )
    estimator.fit(fit_vectors)
    stored_codebook = estimator.cluster_centers_.astype(codebook_dtype)
    assignments = pairwise_distances_argmin(
        flat_vectors, stored_codebook, metric="euclidean"
    ).reshape(vectors.shape[:2]).astype(assignment_dtype(codebook_size))
    return CodebookQuantizedTensor(
        name=name,
        assignments=assignments,
        codebooks=stored_codebook,
        layout=layout,
        mode="vector",
        original_dtype=str(np.asarray(array).dtype),
        assignment_bits=max(1, ceil(log2(codebook_size))),
    )

def sklearn_codebook_vectorize_model(keras_model):
    tensors = []
    skipped = []
    original_bytes = 0
    compressed_bytes = 0
    for tensor_index, weight in enumerate(keras_model.weights):
        values = weight.numpy()
        name = getattr(weight, "path", weight.name)
        original_bytes += values.nbytes
        if (not np.issubdtype(values.dtype, np.floating) or values.ndim < 2
                or name in PRESERVED_TENSOR_NAMES):
            skipped.append(name)
            compressed_bytes += values.nbytes
            continue
        result = sklearn_vector_quantize_array(
            values,
            name=name,
            vector_dim=VECTOR_DIM,
            codebook_size=CODEBOOK_SIZE,
            codebook_dtype=CODEBOOK_DTYPE,
            max_samples=MAX_KMEANS_SAMPLES,
            iterations=KMEANS_ITERATIONS,
            tolerance=KMEANS_TOLERANCE,
            seed=RANDOM_SEED + tensor_index,
        )
        tensors.append(result)
        compressed_bytes += result.estimated_compressed_size_bytes
    return ModelCodebookQuantizationResult(
        tensors=tensors,
        skipped_tensor_names=skipped,
        original_size_bytes=original_bytes,
        estimated_compressed_size_bytes=compressed_bytes,
    )

In [5]:
def sensitivity_batches():
    count = min(NUM_IMPORTANCE_CALIBRATION_SAMPLES, len(train_images))
    for start_index in range(0, count, IMPORTANCE_BATCH_SIZE):
        stop = min(start_index + IMPORTANCE_BATCH_SIZE, count)
        yield preprocess_images(train_images[start_index:stop]), train_labels[start_index:stop]

sensitivity_tensor_names = {
    getattr(weight, "path", weight.name) for weight in model.weights
    if len(weight.shape) >= 2
    and getattr(weight, "path", weight.name) not in PRESERVED_TENSOR_NAMES
}
gradient_importance = estimate_gradient_importance(
    model, sensitivity_batches(),
    included_tensor_names=sensitivity_tensor_names, use_square_root=True
)
start = time.perf_counter()
sklearn_result = sklearn_codebook_vectorize_model(model)
sklearn_quantization_seconds = time.perf_counter() - start

sklearn_model = keras.models.load_model(MODEL_PATH, compile=False)
sklearn_model.set_weights(
    reconstruct_codebook_model_weights(model, sklearn_result)
)
print(f"scikit-learn codebook construction: {sklearn_quantization_seconds:.2f} s")
print(f"Quantized tensors: {len(sklearn_result.tensors)}")

scikit-learn codebook construction: 12.24 s
Quantized tensors: 19


## 4. Custom mathematical codebook vectorization

This path uses the project's explicit NumPy k-means++ and nearest-codeword mathematics with bounded calibration-gradient sensitivity. It uses the same true-W8 scalar representation, tensor selection, FP32 codebook storage, and preserved first/final kernels as the library baseline.

In [6]:
start = time.perf_counter()
custom_result = codebook_vectorize_model(
    model,
    vector_dim=VECTOR_DIM,
    codebook_size=CODEBOOK_SIZE,
    mode="vector",
    quantize_min_rank=2,
    excluded_tensor_names=PRESERVED_TENSOR_NAMES,
    importance_by_name=gradient_importance,
    codebook_dtype=CODEBOOK_DTYPE,
    max_kmeans_samples=MAX_KMEANS_SAMPLES,
    kmeans_iterations=KMEANS_ITERATIONS,
    seed=RANDOM_SEED,
)
custom_quantization_seconds = time.perf_counter() - start

custom_model = keras.models.load_model(MODEL_PATH, compile=False)
custom_model.set_weights(
    reconstruct_codebook_model_weights(model, custom_result)
)
gc.collect()
print(f"Custom codebook construction: {custom_quantization_seconds:.2f} s")
print(f"Quantized tensors: {len(custom_result.tensors)}")

Custom codebook construction: 49.73 s
Quantized tensors: 19


## 5. Tensor coverage, reconstruction error, and memory

In [7]:
original_by_name = {
    getattr(weight, "path", weight.name): weight.numpy()
    for weight in model.weights
}
sklearn_by_name = {tensor.name: tensor for tensor in sklearn_result.tensors}
custom_by_name = {tensor.name: tensor for tensor in custom_result.tensors}
comparison_rows = []
for name, sklearn_tensor in sklearn_by_name.items():
    custom_tensor = custom_by_name[name]
    original = original_by_name[name]
    comparison_rows.append({
        "tensor": name,
        "rank": original.ndim,
        "shape": tuple(original.shape),
        "values": original.size,
        "sklearn_mse": quantization_mse(original, sklearn_tensor),
        "custom_mse": quantization_mse(original, custom_tensor),
        "sklearn_compressed_bytes": sklearn_tensor.estimated_compressed_size_bytes,
        "custom_compressed_bytes": custom_tensor.estimated_compressed_size_bytes,
    })
layer_comparison = pd.DataFrame(comparison_rows)

original_bytes = sklearn_result.original_size_bytes
original_values = int(sum(weight.numpy().size for weight in model.weights))
quantized_values = int(sum(original_by_name[name].size for name in sklearn_by_name))
comparison_summary = pd.DataFrame([
    {"model": "Original FP32 ResNet-18", "codebook_implementation": "none", "quantized_tensors": 0, "preserved_sensitive_tensors": 0, "quantized_values": 0, "parameter_bytes": original_bytes, "quantization_seconds": 0.0},
    {"model": "scikit-learn codebook VQ", "codebook_implementation": "sklearn.cluster.KMeans", "quantized_tensors": len(sklearn_result.tensors), "preserved_sensitive_tensors": len(PRESERVED_TENSOR_NAMES), "quantized_values": quantized_values, "parameter_bytes": sklearn_result.estimated_compressed_size_bytes, "quantization_seconds": sklearn_quantization_seconds},
    {"model": "Custom codebook VQ", "codebook_implementation": "NumPy k-means", "quantized_tensors": len(custom_result.tensors), "preserved_sensitive_tensors": len(PRESERVED_TENSOR_NAMES), "quantized_values": quantized_values, "parameter_bytes": custom_result.estimated_compressed_size_bytes, "quantization_seconds": custom_quantization_seconds},
])
comparison_summary["parameter_mib"] = comparison_summary["parameter_bytes"] / 1024**2
comparison_summary["compression_ratio_vs_fp32"] = original_bytes / comparison_summary["parameter_bytes"]
comparison_summary["memory_reduction_percent"] = (1 - comparison_summary["parameter_bytes"] / original_bytes) * 100
comparison_summary["effective_bits_per_all_parameter_value"] = comparison_summary["parameter_bytes"] * 8 / original_values

display(comparison_summary)
display(layer_comparison.head(10))

,model,codebook_implementation,quantized_tensors,preserved_sensitive_tensors,quantized_values,parameter_bytes,quantization_seconds,parameter_mib,compression_ratio_vs_fp32,memory_reduction_percent,effective_bits_per_all_parameter_value
0,Original FP32 ResNet-18,none,0,0,0,44764968,0.000000,42.69120,1.00000,0.000000,32.00000
1,scikit-learn codebook VQ,sklearn.cluster.KMeans,19,2,11157504,11311912,12.237199,10.78788,3.95733,74.730437,8.08626
2,Custom codebook VQ,NumPy k-means,19,2,11157504,11311912,49.734794,10.78788,3.95733,74.730437,8.08626


,tensor,rank,shape,values,sklearn_mse,custom_mse,sklearn_compressed_bytes,custom_compressed_bytes
0,stack0_block0_1_conv/kernel,4,"(3, 3, 64, 64)",36864,0.000004,0.000005,37888,37888
1,stack0_block0_2_conv/kernel,4,"(3, 3, 64, 64)",36864,0.000004,0.000004,37888,37888
2,stack0_block1_1_conv/kernel,4,"(3, 3, 64, 64)",36864,0.000005,0.000005,37888,37888
3,stack0_block1_2_conv/kernel,4,"(3, 3, 64, 64)",36864,0.000004,0.000005,37888,37888
4,stack1_block0_1_conv/kernel,4,"(3, 3, 64, 128)",73728,0.000005,0.000005,74752,74752
5,stack1_block0_0_conv/kernel,4,"(1, 1, 64, 128)",8192,0.000003,0.000004,9216,9216
6,stack1_block0_2_conv/kernel,4,"(3, 3, 128, 128)",147456,0.000006,0.000007,148480,148480
7,stack1_block1_1_conv/kernel,4,"(3, 3, 128, 128)",147456,0.000005,0.000006,148480,148480
8,stack1_block1_2_conv/kernel,4,"(3, 3, 128, 128)",147456,0.000006,0.000005,148480,148480
9,stack2_block0_1_conv/kernel,4,"(3, 3, 128, 256)",294912,0.000005,0.000006,295936,295936


### Parameter tensors grouped by rank

Both methods quantize the same floating tensors with rank `>=2`; rank-1 bias and BatchNorm tensors remain FP32.

In [8]:
rank_rows = []
for weight in model.weights:
    values = weight.numpy()
    rank_rows.append({
        "tensor": getattr(weight, "path", weight.name),
        "rank": values.ndim,
        "values": values.size,
        "codebook_quantized": getattr(weight, "path", weight.name) in sklearn_by_name,
        "preserved_sensitive": getattr(weight, "path", weight.name) in PRESERVED_TENSOR_NAMES,
    })
rank_details = pd.DataFrame(rank_rows)
rank_summary = rank_details.groupby("rank", as_index=False).agg(
    total_tensors=("tensor", "count"),
    total_values=("values", "sum"),
    quantized_tensors=("codebook_quantized", "sum"),
    preserved_sensitive_tensors=("preserved_sensitive", "sum"),
)
rank_summary

,rank,total_tensors,total_values,quantized_tensors,preserved_sensitive_tensors
0,1,81,19210,0,0
1,2,1,5120,0,1
2,4,20,11166912,19,1


## 6. Predictions and accuracy comparison

The integer codebook assignments are decoded once into reconstructed weights for Keras accuracy evaluation. All operations and activations remain FP32. This measures codebook reconstruction error; it is not an indexed-codebook hardware runtime.

In [9]:
def predict_keras_labels(keras_model, raw_images, batch_size=EVALUATION_BATCH_SIZE):
    predictions = []
    for start in range(0, len(raw_images), batch_size):
        batch = preprocess_images(raw_images[start : start + batch_size])
        logits = keras_model(batch, training=False).numpy()
        predictions.append(np.argmax(logits, axis=1))
    return np.concatenate(predictions)

sample_label = int(test_labels[0])
sample_predictions = {
    "Original FP32": int(predict_keras_labels(model, test_images[:1], 1)[0]),
    "scikit-learn codebook VQ": int(predict_keras_labels(sklearn_model, test_images[:1], 1)[0]),
    "Custom codebook VQ": int(predict_keras_labels(custom_model, test_images[:1], 1)[0]),
}
pd.DataFrame([{
    "model": name,
    "true_class": CIFAR10_CLASS_NAMES[sample_label],
    "predicted_class": CIFAR10_CLASS_NAMES[prediction],
    "correct": prediction == sample_label,
} for name, prediction in sample_predictions.items()])

,model,true_class,predicted_class,correct
0,Original FP32,cat,cat,True
1,scikit-learn codebook VQ,cat,cat,True
2,Custom codebook VQ,cat,cat,True


In [10]:
print("1/3 Evaluating original FP32 model...")
fp32_predictions = predict_keras_labels(model, test_images)
print("2/3 Evaluating scikit-learn codebook model...")
sklearn_predictions = predict_keras_labels(sklearn_model, test_images)
print("3/3 Evaluating custom codebook model...")
custom_predictions = predict_keras_labels(custom_model, test_images)

fp32_accuracy = float(np.mean(fp32_predictions == test_labels))
sklearn_accuracy = float(np.mean(sklearn_predictions == test_labels))
custom_accuracy = float(np.mean(custom_predictions == test_labels))
accuracy_table = pd.DataFrame([
    {"model": "Original FP32 ResNet-18", "codebook": "none", "correct_predictions": int(np.sum(fp32_predictions == test_labels)), "test_images": len(test_labels), "accuracy_percent": fp32_accuracy * 100, "drop_from_fp32_percentage_points": 0.0},
    {"model": "scikit-learn codebook VQ", "codebook": "true W8 scalar KMeans; first/final kernels FP32", "correct_predictions": int(np.sum(sklearn_predictions == test_labels)), "test_images": len(test_labels), "accuracy_percent": sklearn_accuracy * 100, "drop_from_fp32_percentage_points": (fp32_accuracy - sklearn_accuracy) * 100},
    {"model": "Custom codebook VQ", "codebook": "true W8 sensitivity-refined NumPy k-means; first/final kernels FP32", "correct_predictions": int(np.sum(custom_predictions == test_labels)), "test_images": len(test_labels), "accuracy_percent": custom_accuracy * 100, "drop_from_fp32_percentage_points": (fp32_accuracy - custom_accuracy) * 100},
])
accuracy_table

1/3 Evaluating original FP32 model...
2/3 Evaluating scikit-learn codebook model...
3/3 Evaluating custom codebook model...


,model,codebook,correct_predictions,test_images,accuracy_percent,drop_from_fp32_percentage_points
0,Original FP32 ResNet-18,none,9047,10000,90.47,0.00
1,scikit-learn codebook VQ,true W8 scalar KMeans; first/final kernels FP32,9039,10000,90.39,0.08
2,Custom codebook VQ,true W8 sensitivity-refined NumPy k-means; fir...,9042,10000,90.42,0.05


## 7. Save results

In [11]:
accuracy_path = OUTPUT_DIR / "codebook_accuracy_comparison.csv"
summary_path = OUTPUT_DIR / "codebook_memory_timing_comparison.csv"
layer_path = OUTPUT_DIR / "codebook_layer_reconstruction_comparison.csv"
rank_path = OUTPUT_DIR / "parameter_rank_summary.csv"
results_path = OUTPUT_DIR / "results.json"
accuracy_table.to_csv(accuracy_path, index=False)
comparison_summary.to_csv(summary_path, index=False)
layer_comparison.to_csv(layer_path, index=False)
rank_summary.to_csv(rank_path, index=False)
results_path.write_text(json.dumps({
    "model": str(MODEL_PATH),
    "dataset": "CIFAR-10 test",
    "evaluation_images": int(len(test_labels)),
    "vector_dim": VECTOR_DIM,
    "codebook_size": CODEBOOK_SIZE,
    "assignment_bits_per_vector": ASSIGNMENT_BITS,
    "assignment_bits_per_scalar": ASSIGNMENT_BITS / VECTOR_DIM,
    "preserved_fp32_tensor_names": sorted(PRESERVED_TENSOR_NAMES),
    "fp32_accuracy_percent": fp32_accuracy * 100,
    "sklearn_codebook_accuracy_percent": sklearn_accuracy * 100,
    "custom_codebook_accuracy_percent": custom_accuracy * 100,
    "activations_quantized": False,
}, indent=2) + "\n", encoding="utf-8")
print(f"Saved results under: {OUTPUT_DIR}")

Saved results under: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/resnet18_codebook_vq_comparison


## Final codebook vector-quantization accuracy comparison (%)

In [12]:
final_accuracy_table = accuracy_table.copy()
final_accuracy_table["accuracy_percent"] = final_accuracy_table["accuracy_percent"].map(lambda value: f"{value:.2f}%")
final_accuracy_table["drop_from_fp32_percentage_points"] = final_accuracy_table["drop_from_fp32_percentage_points"].map(lambda value: f"{value:.2f}")
final_accuracy_table

,model,codebook,correct_predictions,test_images,accuracy_percent,drop_from_fp32_percentage_points
0,Original FP32 ResNet-18,none,9047,10000,90.47%,0.00
1,scikit-learn codebook VQ,true W8 scalar KMeans; first/final kernels FP32,9039,10000,90.39%,0.08
2,Custom codebook VQ,true W8 sensitivity-refined NumPy k-means; fir...,9042,10000,90.42%,0.05
